<a href="https://colab.research.google.com/github/AditiSinghal28/Machine-Learning-Projects/blob/ml/Fake_News_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importing the dependencies

In [ ]:
import numpy as np
import pandas as pd
import re #regular expression
from nltk.corpus import stopwords #nltk = natural language tool kit...corpus means body..stopwords are the words that dont add much value to the data like a, an, the, in etc.
from nltk.stem.porter import PorterStemmer #stemming func removes the prefix and suffix of the word and returns the root word
from sklearn.feature_extraction.text import TfidfVectorizer #convert text to numerical values
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
print(stopwords.words('english')) #printing stopwords in english

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

Data Preprocessing

In [13]:
#loading dataset into a pandas dataframe
news_dataset = pd.read_csv('/content/WELFake_Dataset.csv')

In [14]:
news_dataset.shape

(72134, 4)

In [15]:
#printing the first five rows of dataframe
news_dataset.head()

,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [16]:
#Counting the number of missing values in data frame
news_dataset.isnull().sum()

,0
Unnamed: 0,0
title,558
text,39
label,0


As we have a large dataset and missing values are less compared to dataset we'll just replace empty values with empty string

In [17]:
news_dataset = news_dataset.fillna('')

In [28]:
#merging title column and text column
news_dataset['content'] = news_dataset['title']+' '+news_dataset['text']
news_dataset.head()

,Unnamed: 0,title,text,label,content
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1,LAW ENFORCEMENT ON HIGH ALERT Following Threat...
1,1,,Did they post their votes for Hillary already?,1,Did they post their votes for Hillary already?
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0,"Bobby Jindal, raised Hindu, uses story of Chri..."
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1,SATAN 2: Russia unvelis an image of its terrif...


In [19]:
#printing new dataframe column
print(news_dataset['content'])

0        LAW ENFORCEMENT ON HIGH ALERT Following Threat...
1           Did they post their votes for Hillary already?
2        UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...
3        Bobby Jindal, raised Hindu, uses story of Chri...
4        SATAN 2: Russia unvelis an image of its terrif...
                               ...                        
72129    Russians steal research on Trump in hack of U....
72130     WATCH: Giuliani Demands That Democrats Apolog...
72131    Migrants Refuse To Leave Train At Refugee Camp...
72132    Trump tussle gives unpopular Mexican leader mu...
72133    Goldman Sachs Endorses Hillary Clinton For Pre...
Name: content, Length: 72134, dtype: object


In [20]:
#separating the data(content) and labels
X = news_dataset.drop(columns='label', axis=1)
Y = news_dataset['label']

In [21]:
print(X)
print(Y)

       Unnamed: 0  ...                                            content
0               0  ...  LAW ENFORCEMENT ON HIGH ALERT Following Threat...
1               1  ...     Did they post their votes for Hillary already?
2               2  ...  UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...
3               3  ...  Bobby Jindal, raised Hindu, uses story of Chri...
4               4  ...  SATAN 2: Russia unvelis an image of its terrif...
...           ...  ...                                                ...
72129       72129  ...  Russians steal research on Trump in hack of U....
72130       72130  ...   WATCH: Giuliani Demands That Democrats Apolog...
72131       72131  ...  Migrants Refuse To Leave Train At Refugee Camp...
72132       72132  ...  Trump tussle gives unpopular Mexican leader mu...
72133       72133  ...  Goldman Sachs Endorses Hillary Clinton For Pre...

[72134 rows x 4 columns]
0        1
1        1
2        1
3        0
4        1
        ..
72129    0
72130    

Stemming :

Stemming is the process of reducing a word to its root word

In [23]:
port_stem = PorterStemmer() #loading func into a variable

In [29]:
def stemming(content) : #function for stemming
  stemmed_content = re.sub('[^a-zA-Z]',' ',content) #sub is substitute..˄ means excluding..we are replacing all the non text characters to space
  stemmed_content = stemmed_content.lower() #converting all to lowercase
  stemmed_content = stemmed_content.split() #splitting all the words to list
  stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
  stemmed_content = ' '.join(stemmed_content) #joining the stemmed words to a string
  return stemmed_content

In [30]:
news_dataset['content'] = news_dataset['content'].apply(stemming)
#applying the stemming function to the content column and saving it in the content column

In [31]:
print(news_dataset['content'])

0        law enforc high alert follow threat cop white ...
1                                post vote hillari alreadi
2        unbeliev obama attorney gener say charlott rio...
3        bobbi jindal rais hindu use stori christian co...
4        satan russia unv imag terrifi new supernuk wes...
                               ...                        
72129    russian steal research trump hack u democrat p...
72130    watch giuliani demand democrat apolog trump ra...
72131    migrant refus leav train refuge camp hungari m...
72132    trump tussl give unpopular mexican leader much...
72133    goldman sach endors hillari clinton presid gol...
Name: content, Length: 72134, dtype: object


In [32]:
#separating data and label
X = news_dataset['content'].values
Y = news_dataset['label'].values

In [33]:
print(X)

['law enforc high alert follow threat cop white blacklivesmatt fyf terrorist video comment expect barack obama member fyf fukyoflag blacklivesmatt movement call lynch hang white peopl cop encourag other radio show tuesday night turn tide kill white peopl cop send messag kill black peopl america one f yoflag organ call sunshin radio blog show host texa call sunshin f ing opinion radio show snapshot fyf lolatwhitefear twitter page p show urg support call fyf tonight continu dismantl illus white snapshot twitter radio call invit fyf radio show air p eastern standard time show caller clearli call lynch kill white peopl minut clip radio show heard provid breitbart texa someon would like refer hannib alreadi receiv death threat result interrupt fyf confer call unidentifi black man said mother f ker start f ing like us bunch ni er takin one us roll said caus alreadi roll gang anyway six seven black mother f cker see white person lynch ass let turn tabl conspir cop start lose peopl state emerg

In [34]:
print(Y)

[1 1 1 ... 0 0 1]


In [35]:
Y.shape

(72134,)

In [36]:
#converting the textual data to numerical data
vectorizer = TfidfVectorizer() #loading vectorizer func into a variable
vectorizer.fit(X)
#Term frequency inverse document frequency
X = vectorizer.transform(X)

In [37]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 13656667 stored elements and shape (72134, 162203)>
  Coords	Values
  (0, 938)	0.019104619426517897
  (0, 1282)	0.017363778513914716
  (0, 2131)	0.052457780993620334
  (0, 2783)	0.020231302394732035
  (0, 3614)	0.029904475345647965
  (0, 3999)	0.02747310458208844
  (0, 4264)	0.023865946576073604
  (0, 4335)	0.05055646943154232
  (0, 4846)	0.01513932938772633
  (0, 4862)	0.02486341752399553
  (0, 6013)	0.014596932940161228
  (0, 6507)	0.057303304534120254
  (0, 6845)	0.01589099748716943
  (0, 8437)	0.12657603668480968
  (0, 8976)	0.015516676506767142
  (0, 10478)	0.06692816334064453
  (0, 11430)	0.018962618491219916
  (0, 12727)	0.01580162854760987
  (0, 14072)	0.018345912143817013
  (0, 14679)	0.01785037970922704
  (0, 15442)	0.19279395985841352
  (0, 15499)	0.08125624068348719
  (0, 15611)	0.0888918729364855
  (0, 15886)	0.029332963149593518
  (0, 18063)	0.10843561013885229
  :	:
  (72133, 132638)	0.031715743461707
  (72133

Splitting the data into training and test data

In [38]:
X_train, X_test,Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=2)

Training the model

In [39]:
model = LogisticRegression()

In [40]:
model.fit(X_train, Y_train)

LogisticRegression()

EVALUATION

In [42]:
#accuracy score on training data
X_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score(X_train_prediction, Y_train)

In [43]:
print("Accuracy score of training data", training_data_accuracy)

Accuracy score of training data 0.9625695322924429


In [44]:
#accuracy score on test data
X_test_prediction = model.predict(X_test)
test_data_accuracy = accuracy_score(X_test_prediction, Y_test)

In [45]:
print("Accuracy score of test data", test_data_accuracy)

Accuracy score of test data 0.9478061967144936


Making a predictive system

In [46]:
X_new = X_test[0]

prediction = model.predict(X_new)
print(prediction)

if (prediction[0]==0) :
  print("News is fake")
else :
  print("News is real")

[1]
News is real


In [48]:
print(Y_test[0])

1
